# 03 · Zadanie I3: aktywne uczenie z losowym wyborem obrazów

Cel: wytrenować studenta Cellpose-SAM (start: encoder SAM, losowa głowica, czyli model B z notebooka 02)
na obrazach wybieranych **losowo**, aż osiągnie wynik wytrenowanej wyroczni. Losowy wybór to
obowiązkowy baseline dla każdego selektora AL (IT2 FLS, MC-Dropout, Coreset, VAAL).

Symulacja AL:
- pula: 30 obrazów train (`configs/monuseg_split.json`); walidacja: 7 obrazów val; test nietknięty do samego końca,
- „adnotator” = wyrocznia `cpsam`: student uczy się na **maskach wyroczni**, a GT służy tylko do ewaluacji,
- w każdej rundzie dochodzi `step` losowych obrazów (zbiory są zagnieżdżone, kolejność ustala seed),
- w każdej rundzie student jest trenowany **od nowa** z inicjalizacji SAM (cold start, tak samo dla wszystkich selektorów),
- stop: PQ studenta na val ≥ PQ wyroczni na val albo koniec puli.

Kod: `src/histopathology_pnw/al/random_loop.py`. Uruchamiaj na GPU (Ubuntu/CUDA). Wyniki każdej rundy
lądują w `outputs/al_random/...`; ponowne uruchomienie pomija ukończone rundy.

## Jak działa Cellpose-SAM (przegląd implementacji, cellpose 4.2)

**Architektura** (`cellpose/vit.py`, klasa `CPSAM`):
- encoder SAM ViT-L (24 bloki transformera, ~300 M parametrów) przerobiony na wejście 256×256 px:
  patche 8×8 zamiast 16×16 (wagi konwolucji patch-embed podpróbkowane co 2), pozycje 64×64 → 32×32,
  attention globalne we wszystkich blokach zamiast okien,
- „neck” SAM (256 kanałów) + konwolucja 1×1 `out` → 3 × 8 × 8 kanałów na token, rozłożone transponowaną
  konwolucją (`W2`, stała) z powrotem do rozdzielczości pikseli,
- 3 wyjścia na piksel: **flow Y, flow X** (gradient „dyfuzji” od środka komórki) i **cellprob** (logit „piksel należy do komórki”).

**Trening** (`cellpose/train.py`, `train_seg`):
- maski instancji są zamieniane na flow fields (`dynamics.labels_to_flows`),
- w każdej epoce `nimg_per_epoch` losowych wycinków 256×256 z augmentacją (`transforms.random_rotate_and_resize`: obrót, skala, odbicie),
- strata `_loss_fn_seg`: MSE na flow (×5) / 2 + BCE na cellprob,
- AdamW, LR z rozgrzewką przez 10 epok, przy ≥ 100 epokach spadki LR na końcu.

**Inferencja** (`CellposeModel.eval` → `dynamics.compute_masks`): obraz jest dzielony na nachodzące kafle 256 px,
piksele z cellprob > próg „płyną” wzdłuż flow do wspólnego punktu zbieżności, a piksele z tego samego punktu tworzą jedną instancję.
Maski niespójne z flow (`flow_threshold`) i mniejsze niż `min_size` są odrzucane.

Wniosek dla AL: student ma 0 wiedzy o jądrach (notebook 02, model B). Głowica jest losowa, więc nawet
kilka obrazów musi nauczyć ją flow fields od zera. Dlatego budżet treningu na rundę jest stały (`n_epochs × nimg_per_epoch`).

## Konfiguracja

- `MODE = "smoke"`: 2 → 4 obrazy, 1 epoka, 2 obrazy val. Sprawdza tylko, że wszystko działa (minuty).
  PQ studenta wyjdzie 0: w 1. epoce LR jest jeszcze na rozgrzewce (= 0).
- `MODE = "full"`: pełny eksperyment. Na jednym GPU klasy RTX 4090 / A100 runda to rząd kilkunastu minut,
  a cały przebieg kilka godzin. Budżet na rundę: 200 epok × 64 wycinki = 12 800 wycinków (1 600 kroków przy batch 8),
  taki sam w każdej rundzie. Parametry nie były strojone. Jeśli po pierwszych rundach PQ stoi blisko 0, budżet
  jest za mały dla głowicy uczonej od zera: zwiększ `n_epochs` (wybór tylko na podstawie val, nigdy testu).

In [ ]:
MODE = "smoke"  # "smoke" albo "full"
SEEDS = [0]  # docelowo co najmniej 3 seedy, np. [0, 1, 2]

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from cellpose import io as cp_io

from histopathology_pnw.al.random_loop import RandomALConfig, round_sizes, run_random_al
from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg
from histopathology_pnw.device import get_device
from histopathology_pnw.inference import predict
from histopathology_pnw.metrics import evaluate_all, summarize
from histopathology_pnw.models.oracle import oracle_masks
from histopathology_pnw.models.student import load_student

cp_io.logger_setup()  # logi treningu Cellpose (loss co epokę)
device = get_device()
print("device:", device)

if MODE == "smoke":
    configs = {
        s: RandomALConfig(
            seed=s,
            initial=2,
            step=2,
            max_labeled=4,
            n_epochs=1,
            nimg_per_epoch=2,
            batch_size=1,
            stop_at_oracle=False,
        )
        for s in SEEDS
    }
else:
    configs = {
        s: RandomALConfig(
            seed=s,
            initial=2,
            step=2,
            max_labeled=None,
            n_epochs=200,
            nimg_per_epoch=64,
            batch_size=8,
        )
        for s in SEEDS
    }
configs[SEEDS[0]]

In [ ]:
split = monuseg.load_train_val_split()
by_id = monuseg.samples_by_id("train")
pool = [by_id[i] for i in split["train"]]
val = [by_id[i] for i in split["val"]]
if MODE == "smoke":
    val = val[:2]
print(f"pula: {len(pool)} obrazów, val: {len(val)} obrazów")
print("rundy (liczba oznaczonych obrazów):", round_sizes(configs[SEEDS[0]], len(pool)))

## Poprzeczka: wyrocznia na val

PQ wyroczni względem GT na tych samych obrazach val. Notebook 02 zapisuje to w `outputs/i1/oracle_val_summary.json`;
tutaj liczymy na nowo (maski wyroczni są w cache, więc to szybkie), żeby notebook działał też samodzielnie
i w trybie smoke (inny podzbiór val).

In [ ]:
oracle_val = summarize(
    evaluate_all([monuseg.read_mask(s) for s in val], oracle_masks(val), [s.image_id for s in val])
)
ORACLE_VAL_PQ = oracle_val["pq"]
print(f"PQ wyroczni na val: {ORACLE_VAL_PQ:.4f}")
oracle_val

## Pętla AL

Najpierw (tylko raz) wyrocznia oznacza całą pulę (cache w `outputs/oracle_masks/`), potem kolejne rundy.
Przerwany przebieg można wznowić, uruchamiając tę komórkę ponownie.

In [ ]:
results = {}
for seed, cfg in configs.items():
    out = Settings().outputs_dir / "al_random" / MODE / f"seed{seed}"
    results[seed] = run_random_al(
        cfg, pool, val, out, oracle_val_pq=ORACLE_VAL_PQ, device=device
    ).assign(seed=seed)
results = pd.concat(results.values(), ignore_index=True)
results[
    [
        "seed",
        "n_labeled",
        "val_pq",
        "val_sq",
        "val_dq",
        "val_aji",
        "final_train_loss",
        "train_time_s",
    ]
].round(4)

## Krzywa uczenia: PQ studenta vs liczba oznaczonych obrazów

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for metric, a in zip(("val_pq", "val_aji"), ax, strict=True):
    for seed, g in results.groupby("seed"):
        a.plot(g["n_labeled"], g[metric], "o-", label=f"student, seed {seed}")
    a.axhline(oracle_val[metric.removeprefix("val_")], color="k", ls="--", label="wyrocznia")
    a.set_xlabel("liczba oznaczonych obrazów")
    a.set_ylabel(metric.removeprefix("val_").upper() + " (val)")
    a.legend()
plt.tight_layout()
fig.savefig(Settings().outputs_dir / "al_random" / MODE / "learning_curve.png", dpi=120)

In [ ]:
reached = results[results["val_pq"] >= ORACLE_VAL_PQ]
if len(reached):
    print("pierwsza runda, w której student dorównał wyroczni:")
    print(reached.groupby("seed")["n_labeled"].min())
else:
    print(
        f"student nie osiągnął PQ wyroczni ({ORACLE_VAL_PQ:.4f}); najlepiej: {results['val_pq'].max():.4f}"
    )

## Końcowa ewaluacja na teście (jeden raz)

Najlepsza runda wybrana **na val**, potem jedna ewaluacja na 14 obrazach testowych, obok wyroczni.
Test nie wpływa na żaden wybór.

In [ ]:
best = results.loc[results["val_pq"].idxmax()]
print(
    f"najlepsza runda: seed {best['seed']}, {best['n_labeled']} obrazów, val PQ {best['val_pq']:.4f}"
)

test = monuseg.list_split("test")
test_gt = [monuseg.read_mask(s) for s in test]
test_ids = [s.image_id for s in test]
student = load_student(Path(best["checkpoint"]), device)
student_test = summarize(
    evaluate_all(
        test_gt, predict(student, [monuseg.read_image(s.image_path) for s in test]), test_ids
    )
)
oracle_test = summarize(evaluate_all(test_gt, oracle_masks(test), test_ids))
final = pd.DataFrame({"student": student_test, "wyrocznia": oracle_test}).T
final.to_csv(Settings().outputs_dir / "al_random" / MODE / "final_test.csv")
final.round(4)

## Jak czytać wyniki

- Oś X to koszt adnotacji (liczba obrazów oznaczonych przez wyrocznię), oś Y to jakość studenta względem GT.
  Lepszy selektor AL to krzywa przesunięta w lewo: ten sam PQ przy mniejszej liczbie obrazów.
- Student uczy się na maskach wyroczni, więc jej PQ jest naturalnym sufitem. Może go przekroczyć tylko przypadkiem
  (szum, inne błędy niż nauczyciel).
- Jeden seed to za mało, żeby porównywać selektory: docelowo `SEEDS = [0, 1, 2]` (albo 5) i średnia ± odchylenie.
- Hiperparametry (`n_epochs`, `nimg_per_epoch`, `learning_rate`) są wspólne dla wszystkich rund i selektorów.
  Jeśli zmieniasz je, to tylko na podstawie val, nigdy testu.